In [1]:
##########################################################
#                 DATA RETRIEVAL VIA STEAM API
#
# NOTE:
# Running this section may take a significant amount of
# time due to API rate limits and the large number of
# requests required. A delay mechanism is implemented to
# prevent overloading the Steam API.
#
# This dataset extraction covers 100 Steam games in a
# single batch. Previously this was split into two
# batches of 50 due to:
#
# - API timeouts
# - Large-scale data volume
# - Approximately 40% of raw data being unusable after
#   preprocessing
#
# The two batches have been merged into one unified
# pipeline to simplify the workflow and eliminate the
# duplicate reviews that arose from running two separate
# scripts appending to the same output file.
#
# Please refer to the project report for full details.
##########################################################

##########################################################
#                 STEAM API LIMITATIONS
#               AND IMPLEMENTED WORKAROUNDS
#
# The Steam Review API is a powerful publicly available
# data source, but it imposes several constraints that
# affect large-scale data extraction.
# The following limitations were encountered and addressed
# in this pipeline:
#
# 1. REVIEW LIMIT PER REQUEST
# ------------------------------------------------
# Each API request returns a maximum of 100 reviews.
# Therefore, pagination and multiple requests are
# required to collect sufficient data per game.
#
#
# 2. PAGINATION SYSTEM (CURSOR-BASED)
# ------------------------------------------------
# The API uses a cursor to navigate between pages of
# reviews. This ensures sequential access to review
# data without duplication.
#
#
# 3. RATE LIMITING
# ------------------------------------------------
# Excessive request frequency can lead to throttling
# or failed responses from the Steam servers.
#
# To mitigate this, controlled delays were introduced
# between requests using:
#
# time.sleep()
#
#
# 4. LIMITED FILTERING CAPABILITIES
# ------------------------------------------------
# The API only supports basic filtering options such as:
#
# - Language selection (English only)
# - Recent reviews
# - Purchase type (Steam only)
#
# 5. REVIEW IMBALANCE ACROSS GAMES
# ------------------------------------------------
# Some games contain significantly more positive reviews
# than negative reviews, making it necessary to enforce
# class balancing during extraction.
#
#
# 6. MISSING OR EMPTY REVIEWS
# ------------------------------------------------
# Certain API responses contain empty review fields,
# which are removed during processing to maintain
# dataset quality.
#
# 7. DUPLICATE REVIEWS
# ------------------------------------------------
# Appending to the same CSV across multiple runs or
# sessions can introduce duplicate reviews. To prevent
# this, save_to_csv() checks existing review_id values
# (Steam's recommendationid) before writing, and
# balance_dataset() deduplicates by review text before
# sampling.
##########################################################

##########################################################
#                 EXTRACTED DATA FEATURES
#
# Each review extracted from the Steam API contains:
#
# 1. review_id      → Steam's unique review identifier
#                      (recommendationid field from API)
# 2. app_id         → Unique game identifier
# 3. review_text    → User-generated review content
# 4. sentiment      → Binary label (1 = positive,
#                      0 = negative via voted_up field)
# 5. helpful_score  → Community weighted usefulness score
#
# NOTE:
# The timestamp field was intentionally removed as it is
# not required for sentiment classification tasks.
#
# The helpful_score is retained for potential future use
# in quality filtering or analysis.
##########################################################

##########################################################
#                 EXTRACTION PROCESS
#
# The extraction pipeline follows a structured loop-based
# approach:
#
# 1. Iterate through 100 selected Steam App IDs
# 2. Send API request for review data
# 3. Extract relevant fields from JSON response
# 4. Store valid reviews into structured dictionaries
# 5. Use cursor-based pagination to access additional pages
# 6. Repeat until:
#    - Target number of positive/negative reviews is met
#    - No more review pages exist
#
# This ensures large-scale, controlled, and structured
# data collection for  our NLP model training.
##########################################################

##########################################################
#                 FINAL OUTPUT STRUCTURE
#
# The extracted dataset is stored in CSV format:
#
# steam_reviews_final.csv
#
# Each record contains:
#
# review_id | app_id | review_text | sentiment | helpful_score
#
# This dataset is later:
# - Cleaned (noise removal, filtering)
# - Balanced (equal class distribution)
# - Used for model training:
#   * Naive Bayes (TF-IDF)
#   * LSTM (sequence modeling)
#   * BERT (transformer-based classification)
##########################################################

# -*- coding: utf-8 -*-
import sys
import io
import requests
import time
import csv
import os
import pandas as pd
from typing import List, Dict

# ========== FIX UTF-8 DISPLAY IN COLAB ==========
# Force stdout to use UTF-8 (fixes â™¥ garbage in console output)
if hasattr(sys.stdout, 'buffer'):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

# ========== CONFIGURATION ==========
# 100 verified Steam App IDs (mix of paid and free games with many negative reviews)
# Originally extracted in two batches of 50. Merged here into a single pipeline
# to prevent duplicate reviews from repeated CSV appending across separate runs.
APP_IDS = [
    # --- Batch 1 (original 50) ---
    252490, 1245620, 1086940, 814380, 1966720, 1003590, 1551360, 582010, 1091500,
    1174180, 271590, 359550, 381210, 578080, 730, 570, 1172470, 230410, 236390,
    105600, 413150, 892970, 526870, 1145360, 1794680, 550, 322170, 431960, 440,
    218620, 394360, 377160, 289070, 307690, 374320, 637090, 444090, 264710, 362890,
    282140, 601150, 1850570, 1599340, 1332010, 1468270, 1250410, 1361210, 1284190,
    427520,

    # --- Batch 2 (additional 50) ---
    489830, 292030, 620, 400, 220, 546560, 4000, 252950, 1085660, 945360,
    227300, 270880, 782330, 379720, 8870, 346110, 242760, 1326470, 367520, 504230,
    588650, 268910, 848450, 1687950, 12210, 20920, 20900, 570940, 335300, 22380,
    22370, 1030840, 1030830, 1659040, 863550, 870780, 412020, 286690, 1888930,
    205100, 403640, 883710, 2050650, 524220, 646570, 812140, 552520, 1426210
]

# Remove any accidental duplicates across the two batches
APP_IDS = list(dict.fromkeys(APP_IDS))

TARGET_POSITIVE = 500      # target positive reviews per game
TARGET_NEGATIVE = 500      # target negative reviews per game
REVIEWS_PER_PAGE = 100     # Steam API max per request
LANGUAGE = "english"
OUTPUT_CSV = "steam_reviews_final.csv"
REQUEST_DELAY = 0          # seconds between page requests (increase if throttled)

# ========== FUNCTIONS ==========

def fetch_reviews_page(app_id: int, cursor: str = "*") -> dict:
    """Fetch one page of reviews from Steam API with proper UTF-8 handling."""
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    params = {
        "json": 1,
        "language": LANGUAGE,
        "num_per_page": REVIEWS_PER_PAGE,
        "cursor": cursor,
        "filter": "recent",
        "purchase_type": "steam"
    }
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept-Charset': 'utf-8'
    }
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.encoding = 'utf-8'  # Force UTF-8 decoding
        if response.status_code == 200:
            return response.json()
        else:
            print(f"  Error {response.status_code} for app {app_id}")
            return None
    except Exception as e:
        print(f"  Request failed: {e}")
        return None


def collect_reviews_for_app(app_id: int, target_pos: int, target_neg: int) -> List[dict]:
    """Collect reviews for a single game until targets are met or no more reviews."""
    print(f"\n--- Starting App ID: {app_id} ---")
    collected = []
    pos_count = 0
    neg_count = 0
    cursor = "*"
    max_loops = 1000  # Allow deep pagination

    for loop in range(max_loops):
        if pos_count >= target_pos and neg_count >= target_neg:
            break

        print(f"  Loop {loop+1}: pos {pos_count}/{target_pos}, neg {neg_count}/{target_neg}")

        data = fetch_reviews_page(app_id, cursor)
        if not data or "reviews" not in data or not data["reviews"]:
            print("  No more reviews found.")
            break

        for review in data["reviews"]:
            text = review.get("review", "").strip()
            if not text:
                continue
            recommended = review.get("voted_up", False)

            # Only collect if we still need this sentiment class
            if recommended and pos_count < target_pos:
                collected.append({
                    "review_id": review.get("recommendationid", ""),
                    "app_id": app_id,
                    "review_text": text,
                    "sentiment": 1,
                    "helpful_score": review.get("weighted_vote_score", 0),
                })
                pos_count += 1
            elif not recommended and neg_count < target_neg:
                collected.append({
                    "review_id": review.get("recommendationid", ""),
                    "app_id": app_id,
                    "review_text": text,
                    "sentiment": 0,
                    "helpful_score": review.get("weighted_vote_score", 0),
                })
                neg_count += 1

        # Get cursor for next page
        cursor = data.get("cursor", "")
        if not cursor:
            print("  No cursor for next page.")
            break

        time.sleep(REQUEST_DELAY)

    print(f"  Finished app {app_id}: {pos_count} pos, {neg_count} neg")
    return collected


def save_to_csv(all_reviews: List[dict], filename: str):
    """
    Append reviews to CSV with UTF-8-sig encoding (Excel-friendly).

    Deduplication check: before writing, any review whose review_id already
    exists in the file is skipped. Using review_id (Steam's recommendationid)
    is more reliable than text matching, as two users could theoretically
    write identical short reviews. This prevents duplicate rows from
    accumulating if the script is re-run or interrupted and restarted.
    """
    file_exists = os.path.isfile(filename)
    existing_texts = set()

    # Load existing review IDs to avoid writing duplicates
    if file_exists:
        existing = pd.read_csv(filename, encoding='utf-8-sig')
        existing_texts = set(existing['review_id'].astype(str).tolist())

    # Filter out reviews already present in the file
    new_reviews = [r for r in all_reviews if str(r['review_id']) not in existing_texts]
    skipped = len(all_reviews) - len(new_reviews)
    if skipped > 0:
        print(f"  Skipped {skipped} duplicate reviews before saving.")

    with open(filename, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=["review_id", "app_id", "review_text", "sentiment", "helpful_score"])
        if not file_exists:
            writer.writeheader()
        writer.writerows(new_reviews)


def balance_dataset(csv_path: str, output_path: str, target_total: int = 50000):
    """
    Balance the dataset to exactly target_total (half positive, half negative).

    Deduplication is applied before sampling to ensure the balanced output
    contains only unique reviews. This prevents inflated model performance
    caused by identical reviews appearing in both train and test splits.
    """
    df = pd.read_csv(csv_path, encoding='utf-8-sig')

    # Remove duplicate review texts before balancing
    before = len(df)
    df = df.drop_duplicates(subset=['review_text'])
    print(f"  Duplicates removed before balancing: {before - len(df)}")

    pos = df[df['sentiment'] == 1]
    neg = df[df['sentiment'] == 0]
    min_count = min(len(pos), len(neg))
    desired = target_total // 2
    sample_size = min(min_count, desired)

    pos_sample = pos.sample(sample_size, random_state=42)
    neg_sample = neg.sample(sample_size, random_state=42)
    balanced = pd.concat([pos_sample, neg_sample]).sample(frac=1, random_state=42)
    balanced.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"\n=== Balanced dataset saved to {output_path} ===")
    print(f"Size: {len(balanced)} reviews ({sample_size} positive, {sample_size} negative)")
    return balanced


# ========== MAIN ==========
def main():
    print(f"Starting extraction for {len(APP_IDS)} games...")
    all_reviews = []

    for i, app_id in enumerate(APP_IDS):
        print(f"\n--- Game {i+1}/{len(APP_IDS)} ---")
        reviews = collect_reviews_for_app(app_id, TARGET_POSITIVE, TARGET_NEGATIVE)
        if reviews:
            save_to_csv(reviews, OUTPUT_CSV)
            all_reviews.extend(reviews)
        else:
            print(f"  No reviews collected for app {app_id}. Skipping.")
        time.sleep(2)  # Delay between games to avoid throttling

    print(f"\nRaw collection complete. Total reviews collected: {len(all_reviews)}")
    pos_total = sum(1 for r in all_reviews if r['sentiment'] == 1)
    neg_total = len(all_reviews) - pos_total
    print(f"Raw positives: {pos_total}, Raw negatives: {neg_total}")

    if len(all_reviews) > 0:
        balance_dataset(OUTPUT_CSV, "steam_reviews_balanced_50k.csv", target_total=50000)
    else:
        print("No reviews collected at all. Check your internet/API.")


if __name__ == "__main__":
    main()


Streaming output truncated to the last 5000 lines.
  Loop 198: pos 500/500, neg 329/500
  Loop 199: pos 500/500, neg 333/500
  Loop 200: pos 500/500, neg 335/500
  Loop 201: pos 500/500, neg 335/500
  Loop 202: pos 500/500, neg 340/500
  Loop 203: pos 500/500, neg 341/500
  Loop 204: pos 500/500, neg 343/500
  Loop 205: pos 500/500, neg 346/500
  Loop 206: pos 500/500, neg 348/500
  Loop 207: pos 500/500, neg 349/500
  Loop 208: pos 500/500, neg 353/500
  Loop 209: pos 500/500, neg 361/500
  Loop 210: pos 500/500, neg 364/500
  Loop 211: pos 500/500, neg 369/500
  Loop 212: pos 500/500, neg 370/500
  Loop 213: pos 500/500, neg 371/500
  Loop 214: pos 500/500, neg 371/500
  Loop 215: pos 500/500, neg 374/500
  Loop 216: pos 500/500, neg 375/500
  Loop 217: pos 500/500, neg 377/500
  Loop 218: pos 500/500, neg 381/500
  Loop 219: pos 500/500, neg 383/500
  Loop 220: pos 500/500, neg 387/500
  Loop 221: pos 500/500, neg 391/500
  Loop 222: pos 500/500, neg 396/500
  Loop 223: pos 500/500,

In [3]:
##########################################################
#                 DATASET CLEANING / PREPROCESSING
#
# This section involves cleaning the dataset extracted
# and merged from the Steam API. Full methodological
# details are provided in the project report.
#
# The dataset is prepared for three NLP model families:
#
# 1. Naive Bayes (Traditional ML)
# 2. LSTM (Deep Learning)
# 3. BERT (Transformer Models)
#
# The raw Steam review dataset contains real-world user
# generated content, which introduces significant noise
# that must be removed before model training.
#
# The following preprocessing steps are applied:
#
# - Corrupted UTF-8 encoding removal (e.g., ♥, â™¥)
# - URL and hyperlink removal (spam / non-informative text)
# - Language filtering using LangDetect (English only)
# - Emoji removal to ensure consistent tokenisation
# - Spam filtering (repeated characters, symbols, unicode spam)
# - Removal of empty or whitespace-only reviews
#
# IMPORTANT NOTE ON LANGUAGE DETECTION:
# Steam's built-in language metadata is not fully reliable.
# Users may have their Steam client set to English while
# writing reviews in another language (e.g., Chinese).
#
# Therefore, LangDetect is used to verify actual review
# language content rather than relying on Steam metadata.
#
# Example issue:
# - Steam labels review as "English" due to client settings
# - Actual review text may be Chinese or mixed language
#
# LangDetect ensures only true English text is retained,
# improving dataset quality and model performance consistency.
##########################################################

!pip install emoji langdetect

import pandas as pd
import re
import emoji
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

# =========================
# 1. LOAD DATASET
# =========================
file_path = "/content/steam_reviews_final.csv"

df = pd.read_csv(file_path, encoding="utf-8-sig")

initial = len(df)

print("Initial dataset:", initial)

# Ensure all review text values are strings
df["review_text"] = df["review_text"].astype(str)

# =========================
# 4. REMOVE CORRUPTED ♥ ENCODING
# =========================
before = len(df)

df = df[
    ~df["review_text"].str.contains(
        r"♥|â™¥",
        regex=True,
        na=False
    )
]

print("Encoding cleanup removed:", before - len(df))

# =========================
# 5. REMOVE LINKS
# =========================
before = len(df)

df = df[
    ~df["review_text"].str.contains(
        r"http\S+|www\.\S+",
        regex=True,
        na=False
    )
]

print("Link removal removed:", before - len(df))

# =========================
# 6. KEEP ONLY ENGLISH (LANG DETECT)
# =========================
def is_english(text):
    try:
        return detect(text) == "en"
    except:
        return False

before = len(df)

df = df[df["review_text"].apply(is_english)]

print("Language filtering removed:", before - len(df))

# =========================
# 7. REMOVE EMOJIS
# =========================
def has_emoji(text):
    return any(char in emoji.EMOJI_DATA for char in text)

before = len(df)

df = df[~df["review_text"].apply(has_emoji)]

print("Emoji removal removed:", before - len(df))

# =========================
# 8. SPAM FILTER
# =========================
spam_patterns = [
    r"(.)\1{4,}",
    r"[⭐★☆]{3,}",
    r"[⣿]{2,}",
    r"^[^\w\s]{3,}$"
]

spam_regex = re.compile("|".join(spam_patterns))

before = len(df)

df = df[
    ~df["review_text"].str.contains(
        spam_regex,
        regex=True,
        na=False
    )
]

print("Spam filtering removed:", before - len(df))

# =========================
# 9. REMOVE EMPTY ROWS
# =========================
before = len(df)

df = df[df["review_text"].str.strip().astype(bool)]

print("Empty rows removed:", before - len(df))

# =========================
# 10. RESET INDEX + ADD REVIEW ID
# =========================
df = df.reset_index(drop=True)

# Remove existing review_id column if it already exists
if "review_id" in df.columns:
    df = df.drop(columns=["review_id"])

df.insert(0, "review_id", range(1, len(df) + 1))

# =========================
# 11. SAVE FINAL FILE
# =========================
output_path = "/content/steam_reviews_final_clean.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("DONE ✔ Dataset fully cleaned")
print("Final rows:", len(df))
print("Total removed:", initial - len(df))

Initial dataset: 94922
Encoding cleanup removed: 6467
Link removal removed: 216
Language filtering removed: 21950
Emoji removal removed: 538


/tmp/ipykernel_3639/148423011.py:138: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~df["review_text"].str.contains(


Spam filtering removed: 1183
Empty rows removed: 0
DONE ✔ Dataset fully cleaned
Final rows: 64568
Total removed: 30354


In [ ]:
from google.colab import files
files.download("/content/steam_reviews_cleaned.csv")

In [4]:
##########################################################
#    With our data cleaned, we Proceeded with
#   balancing which was done here
##########################################################
##########################################################
                 # DATA BALANCING PART 1
# After the review extraction process was completed,
# the dataset was loaded into a pandas DataFrame for
# preprocessing and balancing.
#
# The purpose of this stage is to ensure that the
# dataset contains an equal number of:
#
# - Positive Reviews
# - Negative Reviews
#
# Balanced datasets are important in Machine Learning
# because unbalanced data may bias the AI model toward
# the majority class.

                 # DATA BALANCING PART 2
# The dataset was separated into two groups using
# the sentiment column:
#
# - sentiment = 1 → Positive Reviews
# - sentiment = 0 → Negative Reviews
#
# This allowed the program to independently count
# and process each sentiment category.
#
# The original review totals for both classes were
# then displayed to determine whether balancing
# was required.

                 # DATA BALANCING PART 3
# Since the dataset may contain more positive
# reviews than negative reviews (or vice versa),
# random sampling was used to safely select an
# equal number of reviews from both classes.
#
# The target size was determined using:
#
# min(30000, len(pos), len(neg))
#
# This prevents sampling errors by ensuring that
# the program never attempts to sample more
# reviews than are available in either class.
#
# Random sampling improves fairness and reduces
# dataset bias during Machine Learning training.

                 # DATA BALANCING PART 4
# Once the positive and negative samples were
# selected, both groups were merged together
# into a single balanced dataset.
#
# The dataset was then shuffled randomly to
# ensure that positive and negative reviews
# are mixed evenly throughout the file.
#
# Shuffling prevents the AI model from learning
# patterns based on review ordering.

                 # DATA BALANCING PART 5
# After balancing and shuffling, new review IDs
# were assigned sequentially to ensure that:
#
# - All review IDs are unique
# - No duplicated IDs exist
# - The dataset remains properly structured
#
# Finally, the completed balanced dataset was
# exported into a new CSV file:
#
# steam_reviews_60k_balanced.csv
#
# The final dataset contains:
#
# - 30 000 Positive Reviews
# - 30 000 Negative Reviews
#
# giving a total of 60 000 balanced reviews
# ready for Machine Learning and Sentiment
# Analysis tasks.

##########################################################
import pandas as pd

# =========================
# 1. LOAD CLEAN DATASET
# =========================
file_path = "/content/steam_reviews_final_clean.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

# =========================
# 2. SPLIT BY SENTIMENT
# =========================
pos = df[df["sentiment"] == 1]
neg = df[df["sentiment"] == 0]

print("Original counts:")
print("Positive:", len(pos))
print("Negative:", len(neg))

# =========================
# 3. SAFE SAMPLING (avoid errors)
# =========================
target = min(30000, len(pos), len(neg))

pos_sample = pos.sample(n=target, random_state=42)
neg_sample = neg.sample(n=target, random_state=42)

# =========================
# 4. MERGE + SHUFFLE
# =========================
balanced = pd.concat([pos_sample, neg_sample])
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# =========================
# 5. FIX / REASSIGN REVIEW IDs
# =========================
balanced["review_id"] = range(1, len(balanced) + 1)

# =========================
# 6. SAVE FINAL DATASET
# =========================
output_path = "/content/steam_reviews_60k_balanced.csv"
balanced.to_csv(output_path, index=False, encoding="utf-8-sig")

# =========================
# 7. FINAL OUTPUT
# =========================
print("DONE ✔ Balanced dataset created")
print("Final size:", len(balanced))
print("Each class:", target)

Original counts:
Positive: 30805
Negative: 33763
DONE ✔ Balanced dataset created
Final size: 60000
Each class: 30000


In [6]:
##########################################################
#                 DATASET SPLITTING (TRAIN/TEST)
#
# After balancing and cleaning the Steam review dataset,
# the data is prepared for Machine Learning training and
# evaluation across three NLP model generations:
#
# 1. Naive Bayes (Traditional ML)
# 2. LSTM (Deep Learning)
# 3. BERT (Transformer-Based Models)
#
# To ensure fair evaluation, the dataset is split into
# training and testing sets using a stratified approach.
#
##########################################################

                 # DATASET SPLITTING PART 1
# The final balanced dataset is loaded from CSV format.
#
# This dataset contains:
# - review_text
# - sentiment (binary label: 1 = positive, 0 = negative)
#
# The dataset is already balanced to ensure equal class
# distribution before model training.

                 # DATASET SPLITTING PART 2
# A 70/30 train-test split is performed:
#
# - 70% of data → Training set
# - 30% of data → Testing set
#
# Stratified sampling is used to ensure that both the
# training and test sets maintain the same class
# distribution as the original dataset.
#
# This is critical for unbiased evaluation across all
# NLP models.

                 # DATASET SPLITTING PART 3
# After splitting, both datasets are re-indexed and
# assigned new sequential review IDs to maintain a
# clean structure.
#
# This ensures:
# - No duplicate IDs
# - Consistent dataset formatting
# - Easy traceability during evaluation

                 # DATASET SPLITTING PART 4
# The test dataset is then separated into two versions:
#
# 1. Test set WITH sentiment labels
#    → Used for evaluating model performance
#
# 2. Test set WITHOUT sentiment labels
#    → Used for simulating real-world predictions
#
# This allows both:
# - Supervised evaluation (accuracy, F1-score)
# - Unlabeled inference testing

                 # DATASET SPLITTING PART 5
# Final datasets are exported as CSV files:
#
# - steam_reviews_train.csv
# - steam_reviews_test_WITH_SENTIMENT.csv
# - steam_reviews_test_NO_SENTIMENT.csv
#
# These datasets are then used in the model training
# and evaluation pipeline for:
#
# - Naive Bayes classification (TF-IDF features)
# - LSTM sequence modeling
# - BERT transformer-based classification
#
##########################################################
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# 1. LOAD BALANCED DATASET
# =========================
file_path = "/content/steam_reviews_60k_balanced.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

# =========================
# 2. SPLIT (70 / 30)
# =========================
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["sentiment"]
)

# =========================
# 3. RESET INDEX + IDS
# =========================
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df["review_id"] = range(1, len(train_df) + 1)
test_df["review_id"] = range(1, len(test_df) + 1)

# =========================
# 4. CREATE TEST SET WITHOUT SENTIMENT
# =========================
test_no_sentiment = test_df.drop(columns=["sentiment"])

# =========================
# 5. SAVE FILES
# =========================
train_df.to_csv("/content/steam_reviews_train_FINAL.csv", index=False, encoding="utf-8-sig")

test_df.to_csv(
    "/content/steam_reviews_test_WITH_SENTIMENT_FINAL.csv",
    index=False,
    encoding="utf-8-sig"
)

test_no_sentiment.to_csv(
    "/content/steam_reviews_test_NO_SENTIMENT_FINAL.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 6. OUTPUT INFO
# =========================
print("DONE ✔ Dataset split complete")
print("Train size:", len(train_df))
print("Test size:", len(test_df))

DONE ✔ Dataset split complete
Train size: 42000
Test size: 18000


In [ ]:
##########################################################
#        Downloading all of our Ready CSVs.
##########################################################
from google.colab import files

files.download("/content/steam_reviews_train.csv")
files.download("/content/steam_reviews_test_WITH_SENTIMENT.csv")
files.download("/content/steam_reviews_test_NO_SENTIMENT.csv")
files.download("/content/steam_reviews_final_clean.csv")